# RTV Check-in Image Classifier — Full Pipeline

This notebook runs the complete ML pipeline from data download to model export:
1. **Setup** — Install dependencies
2. **Data Download** — Fetch dataset zip from Google Drive
3. **Task 1** — Data analysis & preparation
4. **Task 2** — Model training & evaluation
5. **Export** — Save trained model to Google Drive

> **Run in Google Colab** for free GPU access. Set Runtime → Change runtime type → T4 GPU.

---
## 0. Setup & Install Dependencies

In [ ]:
!pip install -q ultralytics torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow opencv-python-headless tqdm

In [ ]:
import os
import sys
import torch

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

---
## 1. Download Dataset from Google Drive

**Instructions:**
1. Upload your dataset zip to Google Drive
2. Right-click → Get link → Copy link
3. Paste the sharing URL below

The link should look like: `https://drive.google.com/file/d/XXXXXXX/view?usp=sharing`

In [ ]:
#@title Paste your Google Drive sharing URL here
GDRIVE_URL = ""  #@param {type:"string"}

# === OR mount Google Drive directly ===
# Uncomment the lines below if the zip is already in your Drive
# from google.colab import drive
# drive.mount('/content/drive')
# GDRIVE_ZIP_PATH = "/content/drive/MyDrive/path/to/data.zip"  # <-- edit this

In [ ]:
import zipfile
import shutil
import re

DATA_DIR = "data"
ZIP_PATH = "dataset.zip"


def extract_file_id(url: str) -> str:
    """Extract the file ID from various Google Drive URL formats."""
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',   # /file/d/ID/view
        r'id=([a-zA-Z0-9_-]+)',         # ?id=ID
        r'/d/([a-zA-Z0-9_-]+)',         # /d/ID
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract file ID from URL: {url}")


def download_from_gdrive(url: str, output: str):
    """Download a file from Google Drive using gdown."""
    file_id = extract_file_id(url)
    download_url = f"https://drive.google.com/uc?id={file_id}&export=download"
    print(f"File ID: {file_id}")
    print(f"Downloading to {output} ...")
    !pip install -q gdown
    import gdown
    gdown.download(download_url, output, quiet=False, fuzzy=True)
    print(f"Download complete: {os.path.getsize(output) / 1e6:.1f} MB")


# Download if URL provided, otherwise check for mounted Drive path
if GDRIVE_URL.strip():
    download_from_gdrive(GDRIVE_URL, ZIP_PATH)
elif 'GDRIVE_ZIP_PATH' in dir() and os.path.exists(GDRIVE_ZIP_PATH):
    print(f"Copying from mounted Drive: {GDRIVE_ZIP_PATH}")
    shutil.copy2(GDRIVE_ZIP_PATH, ZIP_PATH)
else:
    print("ERROR: Set GDRIVE_URL or mount Drive and set GDRIVE_ZIP_PATH above.")

In [ ]:
# Extract the zip
if os.path.exists(ZIP_PATH):
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)

    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')

    # Handle common zip structures: zip might contain a root folder
    # Check if 'data/' exists or if there's a single folder we need to rename
    if not os.path.exists(DATA_DIR):
        # Look for a folder that contains the category subdirs
        expected_cats = {'compost', 'pigsty', 'tippytap', 'vsla', 'organic'}
        for item in os.listdir('.'):
            if os.path.isdir(item) and item != DATA_DIR:
                contents = set(os.listdir(item))
                if expected_cats & contents:  # overlap found
                    os.rename(item, DATA_DIR)
                    print(f"Renamed '{item}' -> '{DATA_DIR}'")
                    break

    # Remove __MACOSX if present
    if os.path.exists('__MACOSX'):
        shutil.rmtree('__MACOSX')

    print("\nExtracted categories:")
    for d in sorted(os.listdir(DATA_DIR)):
        if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith('.'):
            count = len(os.listdir(os.path.join(DATA_DIR, d)))
            print(f"  {d:25s}: {count} files")
else:
    print("No zip file found. Run the download cell above first.")

---
## 2. Task 1 — Data Analysis

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Scan all images
records = []
for cat_dir in sorted(Path(DATA_DIR).iterdir()):
    if not cat_dir.is_dir() or cat_dir.name.startswith(('.', '_')):
        continue
    for img_file in sorted(cat_dir.iterdir()):
        if img_file.suffix.lower() not in IMG_EXTENSIONS:
            continue
        try:
            with Image.open(img_file) as img:
                w, h = img.size
            records.append({
                'category': cat_dir.name,
                'filename': img_file.name,
                'path': str(img_file),
                'width': w,
                'height': h,
                'file_size_kb': round(img_file.stat().st_size / 1024, 1),
            })
        except Exception as e:
            print(f"Skipping corrupt file: {img_file} ({e})")

df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
print(f"Categories:   {df['category'].nunique()}")
print(f"\nClass distribution:")
print(df['category'].value_counts().to_string())

In [ ]:
# Class distribution plot
counts = df['category'].value_counts().sort_values()
imbalance = counts.max() / counts.min()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(counts.index, counts.values, color=sns.color_palette('viridis', len(counts)))
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, str(val),
            va='center', fontweight='bold')
ax.set_xlabel('Number of Images')
ax.set_title(f'Class Distribution (Imbalance Ratio: {imbalance:.1f}:1)')
plt.tight_layout()
plt.show()

In [ ]:
# Sample images grid
categories = sorted(df['category'].unique())
fig, axes = plt.subplots(len(categories), 4, figsize=(12, len(categories) * 2.2))

for i, cat in enumerate(categories):
    cat_df = df[df['category'] == cat].sample(n=min(4, len(df[df['category'] == cat])), random_state=42)
    for j in range(4):
        ax = axes[i][j]
        ax.set_xticks([]); ax.set_yticks([])
        if j < len(cat_df):
            img = Image.open(cat_df.iloc[j]['path']).convert('RGB')
            ax.imshow(img)
        if j == 0:
            ax.set_ylabel(cat, fontsize=8, rotation=0, labelpad=70, ha='right')

fig.suptitle('Sample Images per Category', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Image dimension statistics
print("=== Image Dimensions ===")
print(f"Width  — min: {df['width'].min()}, max: {df['width'].max()}, mean: {df['width'].mean():.0f}")
print(f"Height — min: {df['height'].min()}, max: {df['height'].max()}, mean: {df['height'].mean():.0f}")
print(f"File KB — min: {df['file_size_kb'].min()}, max: {df['file_size_kb'].max()}, mean: {df['file_size_kb'].mean():.0f}")
print(f"\n=== Key Observations ===")
print(f"1. Severe imbalance: guinea-pig-shelter has ~{counts.min()} images vs ~{counts.max()} for others")
print(f"2. Imbalance ratio: {imbalance:.1f}:1")
print(f"3. Small dataset (~{len(df)} images) — transfer learning is essential")
print(f"4. Mixed resolutions — need standardised preprocessing")

---
## 3. Task 1 — Data Pipeline (Split + Augmentation)

In [ ]:
import random
import shutil
from sklearn.model_selection import train_test_split
from PIL import ImageEnhance, ImageFilter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = 'prepared_data'
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

# Collect samples
samples = [{'path': r['path'], 'label': r['category']} for _, r in df.iterrows()]
paths = [s['path'] for s in samples]
labels = [s['label'] for s in samples]

print(f"Total samples: {len(samples)}")

In [ ]:
# Stratified split
train_val_p, test_p, train_val_l, test_l = train_test_split(
    paths, labels, test_size=TEST_RATIO, stratify=labels, random_state=SEED)

val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_p, val_p, train_l, val_l = train_test_split(
    train_val_p, train_val_l, test_size=val_frac, stratify=train_val_l, random_state=SEED)

splits = {
    'train': list(zip(train_p, train_l)),
    'val':   list(zip(val_p, val_l)),
    'test':  list(zip(test_p, test_l)),
}

for name, data in splits.items():
    c = Counter(l for _, l in data)
    print(f"{name:5s}: {len(data):4d} images — {dict(sorted(c.items()))}")

In [ ]:
# Copy into YOLO directory structure
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for split_name, data in splits.items():
    for img_path, label in data:
        dest_dir = os.path.join(OUTPUT_DIR, split_name, label)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy2(img_path, os.path.join(dest_dir, os.path.basename(img_path)))

print("Copied originals into train/val/test structure.")

In [ ]:
# Augment minority classes to ~80% of the majority class count
def augment_image(img, idx):
    """Apply one of several augmentations based on index."""
    transforms = [
        lambda x: x.transpose(Image.FLIP_LEFT_RIGHT),
        lambda x: x.rotate(random.uniform(-15, 15), fillcolor=(0,0,0)),
        lambda x: ImageEnhance.Brightness(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Contrast(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Color(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: x.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5))),
        lambda x: x.rotate(random.uniform(-25, 25), fillcolor=(0,0,0)),
    ]
    return transforms[idx % len(transforms)](x=img)

train_counts = Counter(l for _, l in splits['train'])
target = int(max(train_counts.values()) * 0.8)

for label, count in train_counts.items():
    needed = target - count
    if needed <= 0:
        continue
    print(f"Augmenting {label}: +{needed} images (from {count} originals)")
    source_imgs = [p for p, l in splits['train'] if l == label]
    dest_dir = os.path.join(OUTPUT_DIR, 'train', label)

    for i in range(needed):
        src = random.choice(source_imgs)
        img = Image.open(src).convert('RGB')
        aug = augment_image(img, i)
        aug.save(os.path.join(dest_dir, f'aug_{i:04d}_{os.path.basename(src)}'), quality=95)

# Print final counts
print("\nFinal training set:")
for cat_dir in sorted(Path(OUTPUT_DIR, 'train').iterdir()):
    if cat_dir.is_dir():
        print(f"  {cat_dir.name:25s}: {len(list(cat_dir.glob('*')))}")

In [ ]:
# Compute class weights (inverse frequency)
train_labels = [l for _, l in splits['train']]
train_counts = Counter(train_labels)
total = sum(train_counts.values())
n_classes = len(train_counts)

class_weights = {label: total / (n_classes * cnt) for label, cnt in train_counts.items()}
print("Class weights (inverse frequency):")
for label in sorted(class_weights):
    print(f"  {label:25s}: {class_weights[label]:.3f}")

---
## 4. Task 2 — Model Training (YOLO11)

In [ ]:
from ultralytics import YOLO

#@title Training Configuration
MODEL_NAME = 'yolo11n-cls.pt'  #@param ['yolo11n-cls.pt', 'yolo11s-cls.pt', 'yolo11m-cls.pt']
EPOCHS     = 50   #@param {type:"integer"}
IMGSZ      = 640  #@param {type:"integer"}
BATCH      = 32   #@param {type:"integer"}
LR         = 0.001 #@param {type:"number"}
PATIENCE   = 15   #@param {type:"integer"}
TWO_PHASE  = True  #@param {type:"boolean"}

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {DEVICE}")
print(f"Model: {MODEL_NAME}, Epochs: {EPOCHS}, Two-phase: {TWO_PHASE}")

In [ ]:
if TWO_PHASE:
    # Phase 1: Train head only (backbone frozen)
    print("=" * 60)
    print("PHASE 1 — Training classifier head (backbone frozen, 10 epochs)")
    print("=" * 60)
    model = YOLO(MODEL_NAME)
    model.train(
        data=OUTPUT_DIR,
        epochs=10,
        imgsz=IMGSZ,
        batch=BATCH,
        lr0=0.01,
        lrf=0.1,
        patience=10,
        device=DEVICE,
        project='training',
        name='phase1_head',
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        freeze=10,
        warmup_epochs=2,
        cos_lr=True,
        seed=SEED,
        workers=2,
        verbose=True,
    )

    # Phase 2: Fine-tune full model
    print("\n" + "=" * 60)
    print(f"PHASE 2 — Full fine-tuning ({EPOCHS - 10} epochs)")
    print("=" * 60)
    phase1_best = 'training/phase1_head/weights/best.pt'
    model = YOLO(phase1_best)
    model.train(
        data=OUTPUT_DIR,
        epochs=EPOCHS - 10,
        imgsz=IMGSZ,
        batch=BATCH,
        lr0=LR,
        lrf=0.01,
        patience=PATIENCE,
        device=DEVICE,
        project='training',
        name='phase2_finetune',
        exist_ok=True,
        optimizer='AdamW',
        warmup_epochs=3,
        cos_lr=True,
        seed=SEED,
        workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15.0, translate=0.1, scale=0.5,
        fliplr=0.5, mixup=0.1, erasing=0.3,
        verbose=True,
    )
    BEST_MODEL = 'training/phase2_finetune/weights/best.pt'

else:
    # Single-phase training
    print("=" * 60)
    print(f"Single-phase training ({EPOCHS} epochs)")
    print("=" * 60)
    model = YOLO(MODEL_NAME)
    model.train(
        data=OUTPUT_DIR,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        lr0=LR,
        lrf=0.01,
        patience=PATIENCE,
        device=DEVICE,
        project='training',
        name='single_phase',
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        weight_decay=5e-4,
        warmup_epochs=5,
        cos_lr=True,
        seed=SEED,
        workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15.0, translate=0.1, scale=0.5,
        fliplr=0.5, mixup=0.1, erasing=0.3,
        crop_fraction=0.8,
        verbose=True,
    )
    BEST_MODEL = 'training/single_phase/weights/best.pt'

print(f"\nBest model saved at: {BEST_MODEL}")

---
## 5. Task 2 — Evaluation

In [ ]:
# Evaluate on test set
model = YOLO(BEST_MODEL)
results = model.val(split='test')

top1 = results.results_dict.get('metrics/accuracy_top1', 0)
top5 = results.results_dict.get('metrics/accuracy_top5', 0)
print(f"\nTest Set Results:")
print(f"  Top-1 Accuracy: {top1:.4f}")
print(f"  Top-5 Accuracy: {top5:.4f}")

In [ ]:
from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm import tqdm

# Per-image evaluation for detailed metrics
test_dir = Path(OUTPUT_DIR) / 'test'
y_true, y_pred, confidences = [], [], []

class_names = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])

for class_dir in sorted(test_dir.iterdir()):
    if not class_dir.is_dir():
        continue
    true_label = class_dir.name
    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() not in IMG_EXTENSIONS:
            continue
        preds = model.predict(str(img_path), imgsz=IMGSZ, verbose=False)
        pred_label = preds[0].names[preds[0].probs.top1]
        conf = float(preds[0].probs.top1conf)
        y_true.append(true_label)
        y_pred.append(pred_label)
        confidences.append(conf)

print(classification_report(y_true, y_pred, labels=class_names, zero_division=0))
print(f"Macro F1:    {f1_score(y_true, y_pred, labels=class_names, average='macro', zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, labels=class_names, average='weighted', zero_division=0):.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=class_names)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, fmt, title in [
    (axes[0], cm, 'd', 'Confusion Matrix (Counts)'),
    (axes[1], cm_norm, '.2f', 'Confusion Matrix (Normalised)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Confidence distribution
correct = [c for c, t, p in zip(confidences, y_true, y_pred) if t == p]
wrong   = [c for c, t, p in zip(confidences, y_true, y_pred) if t != p]

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 30)
ax.hist(correct, bins=bins, alpha=0.7, color='green', label=f'Correct (n={len(correct)})')
if wrong:
    ax.hist(wrong, bins=bins, alpha=0.7, color='red', label=f'Wrong (n={len(wrong)})')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence Distribution')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Export Model to Google Drive

Copy the best model weights to your Google Drive so you can download them
for local deployment with FastAPI + Docker.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Export destination in Google Drive
DRIVE_EXPORT_DIR = '/content/drive/MyDrive/rtv_classifier'  #@param {type:"string"}

os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

# Copy best model
dest_model = os.path.join(DRIVE_EXPORT_DIR, 'best.pt')
shutil.copy2(BEST_MODEL, dest_model)
print(f"Model exported to: {dest_model}")
print(f"Model size: {os.path.getsize(dest_model) / 1e6:.1f} MB")

# Also copy the last model as backup
last_model = BEST_MODEL.replace('best.pt', 'last.pt')
if os.path.exists(last_model):
    shutil.copy2(last_model, os.path.join(DRIVE_EXPORT_DIR, 'last.pt'))
    print(f"Backup (last.pt) exported too.")

print(f"\nFiles in {DRIVE_EXPORT_DIR}:")
for f in os.listdir(DRIVE_EXPORT_DIR):
    size = os.path.getsize(os.path.join(DRIVE_EXPORT_DIR, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Optional: Download best.pt directly to your local machine (Colab only)
from google.colab import files
files.download(BEST_MODEL)

---
## 7. Next Steps — Local Deployment

After downloading `best.pt` from Google Drive:

```bash
# 1. Place the model in the project
mkdir -p outputs/training
cp ~/Downloads/best.pt outputs/training/best.pt

# 2. Run the FastAPI server
export MODEL_PATH=outputs/training/best.pt
uvicorn src.app.main:app --host 0.0.0.0 --port 8000

# 3. Or deploy with Docker
docker compose up --build

# 4. Open http://localhost:8000 to test
```